# Analyze a directory of results from `curl-speedtest`

## Imports and magic

In [ ]:
import datetime
import json
import statistics
from dataclasses import dataclass, field
from pathlib import Path

import matplotlib.pyplot as plt

%matplotlib widget

## Where your results live; change this to suit.

In [ ]:
result_dir=Path("/rubin/shared/speed-multi")

## Chain of transformation functions.  Only the last three are public.

In [ ]:
def _trim_results(results_dir: Path) -> dict[str, float|str]:
    # Keep the fields we care about
    trimmed_results: dict[str, float|str] = {}
    suffix="-results.json"
    for res in results_dir.glob(f"*{suffix}"):
        f_uuid = res.name[:-len(suffix)]
        r_obj = json.loads(res.read_text())
        # Keep the fields we want
        trimmed_results[f_uuid] = {
            "node": r_obj["node"],
            "start": r_obj["start"],
            "end": r_obj["end"],
            "speed": r_obj["real"]  # Only use wallclock
        }
    return trimmed_results

def _organize_results_by_node(trimmed_results: dict[str, float|str]) -> dict[str,list[dict[str,float]]]:
    # Organize trimmed_results by node
    results_by_node: dict[str,list[dict[str,float]]] = {}
    for v in trimmed_results.values():
        node = v["node"]
        if node not in results_by_node:
            results_by_node[node] = []
        results_by_node[node].append({
            "start": v["start"],
            "end": v["end"],
            "speed": v["speed"]
        })
    return results_by_node

def _sort_node_results(results_by_node: dict[str,list[dict[str,float]]]) -> dict[str,list[dict[str,float]]]:
    # Stack up overlapping intervals
    sorted_results_by_node: dict[str,list[dict[str,float]]] = {}
    for node in results_by_node:
        noderesults = sorted(
            results_by_node[node],
            key=lambda x: x["start"]
        )
        sorted_results_by_node[node] = noderesults    
    return sorted_results_by_node

def _generate_deltas(sorted_results_by_node: dict[str,list[dict[str,float]]]) -> dict[str,dict[float,float]]:
    deltas:dict[str,dict[float,float]] = {}
    for node, s_results in sorted_results_by_node.items():
        if node not in deltas:
            deltas[node] = {}
        interval = deltas[node]
        s_results = sorted_results_by_node[node]
        for res in s_results:
            interval[res["start"]] = res["speed"]
            interval[res["end"]] = -res["speed"]
    return deltas

def _sort_deltas(deltas: dict[str,dict[float,float]]) -> dict[str,dict[float,float]]:
    sorted_deltas:dict[str,dict[float,float]] = {}
    for node, ds in deltas.items():
        if node not in sorted_deltas:
            sorted_deltas[node] = {}
        interval=sorted_deltas[node]
        keylist = sorted(list(ds.keys()))
        for k in keylist:
            interval[k] = ds[k]
    return sorted_deltas


def _generate_speeds(sorted_deltas: dict[str,dict[float,float]]) -> dict[str,dict[float,float]]:
    speeds:dict[str,dict[float,float]] = {}
    current=0
    for node, sds in sorted_deltas.items():
        if node not in speeds:
            speeds[node] = {}
        interval=speeds[node]
        for tstamp, change in sds.items():
            current += change
            interval[tstamp] = current   
    return speeds

def generate_speed_points(speeds: dict[str,dict[float,float]]) -> dict[str,list[tuple[float]]]:
    speed_points:dict[str,list[tuple[float]]] = {}
    for node, spds in speeds.items():
        if node not in speed_points:
            speed_points[node] = []
        interval = speed_points[node]
        for tstamp, speed in spds.items():
            interval.append( (tstamp, speed) )
    return speed_points

def generate_plot_points(speed_points: dict[str,list[tuple[float]]]) -> dict[str,dict[str,list[float]]]:
    plot_points:dict[str,dict[str,list[float]]] = {}
    for node, sps in speed_points.items():
        if node not in plot_points:
            plot_points[node] = {}
        interval = plot_points[node]
        times = [ x[0] for x in sps]
        speeds = [ x[1] for x in sps]
        interval["times"] = times
        interval["speeds"] = speeds
    return plot_points

def make_speed_points(results_dir: Path) -> dict[str,list[tuple[float]]]:
    return generate_speed_points(
        _generate_speeds(
            _sort_deltas(
                _generate_deltas(
                    _sort_node_results(
                        _organize_results_by_node(
                            _trim_results(results_dir)
                        )
                    )
                )
            )
        )
    )

## Set up the plotting function and its helper

In [ ]:
def ts(timestamp: float) -> datetime.datetime:
    return datetime.datetime.fromtimestamp(timestamp, tz=datetime.UTC)

def display_plot_points(plot_points:dict[str,dict[str,list[float]]]) -> None:
    fig,axs = plt.subplots(nrows=len(list(plot_points.keys())), ncols=1, sharex=True, figsize=(9,5))
    nodenum=0
    plt.xticks(rotation=25)
    plt.ylabel("MBPs")
    for node, val in plot_points.items():
        try:
            _ = len(axs)
        except TypeError:
            axs=[axs]
        dts = [ ts(x) for x in val["times"]]
        axs[nodenum].plot(dts, val["speeds"])
        max_speed=max(val["speeds"])
        idx=val["speeds"].index(max_speed)
        date=ts(val["times"][idx])
        max_speed_s='{0:.2f}'.format(max(val["speeds"]))
        print(f"Node {node}: max_speed={max_speed_s} MBps at {date}")
        nodenum += 1

## Plot speeds by node over time

In [ ]:
speed_points=make_speed_points(result_dir)
display_plot_points(generate_plot_points(speed_points))

## Interpolator class to match timestamps across nodes

In [ ]:
@dataclass
class NodePoints:
    """Lists of sample times and speeds at those times for a given node."""
    node: str
    times: list[float]
    speeds: list[float]
    current_index: int=0
    last_time: float = field(init=False)
    first_time: float = field(init=False)

    def __post_init__(self):
        self.first_time = self.times[0]
        self.last_time = self.times[-1]


@dataclass
class NodePointInterpolator:
    """Given a list of NodePoints, ensure that they all have the same timestamps,
    so that we can easily sum over them to get aggregate speeds."""
    node_points: list[NodePoints] = field(default_factory=list)
    current_time: float=0.0
    current_speed: float=0.0

    def _this_entry(self) -> NodePoints|None:
        """Given a current_time, determine which entry it came from.
        If there are multiples, just take the first, because it won't
        matter, as long as we check for identical times in interpolate().
        """
        candidates: list[NodePoints] = []
        for x in self.node_points:
            if x.current_index < len(x.times):
                candidates.append(x)
        if not candidates:
            # We've run out of data
            return None
        for x in candidates:
            if x.times[x.current_index] == self.current_time:
                return x
        raise ValueError(f"No set of node points has {ts(x)}"
                         " as its current time.")

    def _next_time(self) -> float|None:
        """Find the next timestamp across all nodes."""
        candidates: list[NodePoints] = []
        for np in self.node_points:
            if np.current_index < len(np.times):
                candidates.append(np)  # There's still data to consume
        if not candidates:
            # We've run out of data
            return None
        new_time = min(x.times[x.current_index]for x in candidates)
        self.current_time = new_time
        return new_time

    def _interpolate_single_point(self) -> None:
        """For each timestamp across all nodes, ensure that timestamp
        exists on every node.  If it is before or after that node
        began sampling data, the speed at that time is zero.  If it is
        a data point on that node, the speed is whatever the data point
        says it is.  If it is not currently a data point on that node,
        the speed is whatever the last sampled data point for that node
        is.

        Yeah, if I'd used numpy or scipy there's probably already something
        that does this for me.
        """
        current_entry=self._this_entry()
        if current_entry is None:
            return
        for x in self.node_points:          
            if x == current_entry:
                x.current_index += 1
                continue
            if self.current_time < x.first_time:
                x.times.insert(x.current_index, self.current_time)
                x.speeds.insert(x.current_index, 0.0)
                x.current_index += 1
                continue
            if self.current_time > x.last_time:
                # Should we use append() ?  I guess inserting at the end
                # is basically the same, and it makes everything look
                # more regular.
                x.times.insert(x.current_index, self.current_time)
                x.speeds.insert(x.current_index, 0.0)
                x.current_index += 1
                continue
            node_time=x.times[x.current_index]
            if self.current_time > node_time:
                raise ValueError(f"{ts(x.times[x.current_index])}"
                                 f" is later than {ts(node_time)} at index"
                                 f" {x.current_index} for node {x.node}, so something's wrong")
            if self.current_time == node_time:
                # Unlikely, but we have a sample from the same exact time.
                # ...unless of course we've already run interpolate(), in
                # which case, we would absolutely expect this.
                x.current_index += 1
                continue
            # If we made it this far, we should have a previous speed for this node.
            # The speed at this time is the same as what we sampled most recently for
            # this node.
            speed=x.speeds[x.current_index - 1]
            # Pad this entry's list of times and speeds on the left 
            # with this timestamp and the last recorded speed from that entry.
            x.speeds.insert(x.current_index, speed)
            x.times.insert(x.current_index, self.current_time)
            x.current_index += 1
                          
    def interpolate(self) -> None:
        """Loop over all data points, interpolating to each node."""
        while self._next_time():
            self._interpolate_single_point()

    def sum_over_nodes(self) -> list[tuple[float]]:
        """Sum speeds over nodes at each timestamp."""
        # This will only work after interpolation, so we run interpolate() to start.
        # That should be a no-op if we've already run it.
        # The list of times and speeds for each node should all be the same lengths, and all the times
        # ought to be identical across each node.
        # We'll throw an error if any of these assumptions are violated.
        if not self.node_points:
            raise ValueError("No data to aggregate")
        self.interpolate()
        expected_length = len(self.node_points[0].times)
        speed_at_time: dict[float, float] = {}
        timelist: list[float] = []
        for np in self.node_points:
            node = np.node
            if len(np.times) != expected_length:
                raise ValueError(f"Expected {expected_length} times, but found {len(np.times)} for {node}")
            if len(np.speeds) != expected_length:
                raise ValueError(f"Expected {expected_length} speeds, but found {len(np.speeds)} for {node}")
            for idx, time in enumerate(np.times):
                if idx > len(timelist):
                    raise ValueError(f"Time list is {len(timelist)} entries, but we want entry {idx}")
                if idx == len(timelist):
                    timelist.append(time)
                if timelist[idx] != time:
                    raise ValueError(f"Expected {timelist[idx]} at index {idx} in time list, but got {time}")
                if time not in speed_at_time:
                    speed_at_time[time] = 0.0
                speed_at_time[time] += np.speeds[idx]
        return list(zip(speed_at_time.keys(), speed_at_time.values()))
    
def parallelize_speed_points(speed_points: dict[str,list[tuple[float]]]) -> NodePointInterpolator:
    """Ingest speed points and return their interpolator."""
    interpolator=NodePointInterpolator()
    for node in speed_points:
        interpolator.node_points.append(
            NodePoints(
                node=node,
                times=[ x[0] for x in speed_points[node] ],
                speeds=[ x[1] for x in speed_points[node] ]
            )
        )
    
    return interpolator 

## Sum speeds over nodes

In [ ]:
aggregate_speed=parallelize_speed_points(speed_points).sum_over_nodes()

## Plot aggregate speeds over time

In [ ]:
def display_aggregate_speed(plot_points:list[tuple[float]]) -> None:
    """Plot aggregate speeds over time."""
    fig,axs = plt.subplots(figsize=(9,5))
    nodenum=0
    plt.xticks(rotation=25)
    plt.ylabel("MBPs")
    times=[x[0] for x in plot_points]
    speeds=[x[1] for x in plot_points]
    dts = [ts(x) for x in times]
    axs.plot(dts, speeds)
    max_speed=max(speeds)
    idx=speeds.index(max_speed)
    date=ts(times[idx])
    max_speed_s='{0:.2f}'.format(max(speeds))
    print(f"max_speed={max_speed_s} MBps at {date}")
    print(f"median speed: {'{0:.2f}'.format(statistics.median(speeds))} MBps")
    print(f"mean speed: {'{0:.2f}'.format(statistics.mean(speeds))} MBps")

display_aggregate_speed(aggregate_speed)